# FMP ETFs

Read-first examples for FMP-backed ETF prices, metadata, and composition data.

In [ ]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openbb import obb

for env_dir in [Path.cwd(), *Path.cwd().parents]:
    env_path = env_dir / ".env"
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break

from quant_warehouse.ingest.credentials import configure_openbb_credentials
from quant_warehouse.ingest.openbb_fetch import SECTION_ROUTES
from quant_warehouse.platforms.data_providers.fmp.sections import (
    FMP_EXTENDED_EQUITY_SECTIONS,
    FMP_HISTORICAL_EQUITY_SECTIONS,
    FMP_HISTORICAL_ETF_SECTIONS,
)
from quant_warehouse.warehouse import Warehouse
from quant_warehouse.warehouse.sections import (
    CRYPTO_PRICES_LIBRARY,
    CRYPTO_PRICES_SECTION,
    CURRENCY_PRICES_LIBRARY,
    CURRENCY_PRICES_SECTION,
    DEFAULT_CRYPTO_SYMBOLS,
    DEFAULT_CURRENCY_SYMBOLS,
    DEFAULT_ECONOMIC_SERIES,
    DEFAULT_INDEX_SYMBOLS,
    EQUITY_CALENDAR_SECTIONS,
    EQUITY_PRICES_LIBRARY,
    ETF_PRICES_LIBRARY,
    ETF_PRICES_SECTION,
    FUND_PRICES_LIBRARY,
    FUND_PRICES_SECTION,
    INDEX_PRICES_LIBRARY,
    INDEX_PRICES_SECTION,
    MACRO_CALENDAR_LIBRARY,
    MACRO_CALENDAR_SECTION,
    MACRO_CALENDAR_SYMBOL,
    MACRO_ECONOMIC_LIBRARY,
    MACRO_ECONOMIC_SECTION,
    MACRO_RISK_PREMIUM_LIBRARY,
    MACRO_RISK_PREMIUM_SECTION,
    MACRO_TREASURY_LIBRARY,
    MACRO_TREASURY_SECTION,
    MACRO_YIELD_CURVE_LIBRARY,
    MACRO_YIELD_CURVE_SECTION,
    RISK_PREMIUM_SYMBOL,
    TREASURY_BUNDLE_SYMBOL,
    YIELD_CURVE_BUNDLE_SYMBOL,
    fundamental_library,
)
from quant_warehouse.warehouse.storage import provider_library

configure_openbb_credentials()

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

warehouse = Warehouse()
PROVIDER = "fmp"
RUN_REFRESH = False

ETF_SYMBOL = "SPY"

In [ ]:
def preview(frame: pd.DataFrame, rows: int = 5) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    return frame.tail(rows)


def state(symbol: str, section: str, provider: str = PROVIDER) -> dict[str, object] | None:
    item = warehouse.catalog.get(symbol=symbol, section=section, provider=provider)
    return asdict(item) if item is not None else None


def state_table(symbol: str, sections: list[str] | tuple[str, ...], provider: str = PROVIDER) -> pd.DataFrame:
    rows = [state(symbol, section, provider=provider) for section in sections]
    rows = [row for row in rows if row is not None]
    table = pd.DataFrame(rows)
    if not table.empty and "columns_present" in table.columns:
        table.insert(
            table.columns.get_loc("columns_present"),
            "column_count",
            table["columns_present"].map(len),
        )
    return table


def route_table(section_to_base_library: dict[str, str]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "section": section,
                "openbb_route": SECTION_ROUTES.get(section),
                "provider_library": provider_library(base_library, PROVIDER),
            }
            for section, base_library in section_to_base_library.items()
        ]
    )


def run_openbb_route(label: str, fn):
    try:
        result = fn()
        frame = result.to_df()
        return {
            "label": label,
            "provider": getattr(result, "provider", None),
            "rows": 0 if frame is None else len(frame),
            "columns": [] if frame is None else list(frame.columns),
            "error": None,
        }
    except Exception as exc:
        return {
            "label": label,
            "provider": PROVIDER,
            "rows": None,
            "columns": [],
            "error": str(exc),
        }

## Available ETF Sections

In [ ]:
etf_sections = ["etf_prices", "etf_profile", *FMP_HISTORICAL_ETF_SECTIONS]
etf_route_libraries = {
    "etf_prices": ETF_PRICES_LIBRARY,
    "etf_profile": "etf_profile",
    **{section: fundamental_library(section) for section in FMP_HISTORICAL_ETF_SECTIONS},
}
route_table(etf_route_libraries)

## Local Storage Coverage

In [ ]:
state_table(ETF_SYMBOL, etf_sections)

## Prices and Profile

In [ ]:
if RUN_REFRESH:
    warehouse.etf.refresh_prices(ETF_SYMBOL, providers=[PROVIDER])
    warehouse.etf.refresh_profile(ETF_SYMBOL, provider=PROVIDER)

preview(warehouse.etf.read_prices(ETF_SYMBOL, provider=PROVIDER))

In [ ]:
profile = warehouse.etf.read_profile(ETF_SYMBOL, provider=PROVIDER)
asdict(profile) if profile is not None else None

## Holdings and Exposures

In [ ]:
if RUN_REFRESH:
    warehouse.etf.refresh_fundamentals(
        ETF_SYMBOL,
        sections=["etf_holdings", "etf_sectors", "etf_countries", "etf_equity_exposure", "etf_price_performance"],
        providers=[PROVIDER],
    )

{
    "holdings": preview(warehouse.etf.read_fundamentals(ETF_SYMBOL, section="etf_holdings", provider=PROVIDER)),
    "sectors": preview(warehouse.etf.read_fundamentals(ETF_SYMBOL, section="etf_sectors", provider=PROVIDER)),
    "countries": preview(warehouse.etf.read_fundamentals(ETF_SYMBOL, section="etf_countries", provider=PROVIDER)),
    "equity_exposure": preview(warehouse.etf.read_fundamentals(ETF_SYMBOL, section="etf_equity_exposure", provider=PROVIDER)),
}

## Direct FMP/OpenBB Route Check

In [ ]:
pd.DataFrame([
    run_openbb_route("equity.price.historical", lambda: obb.equity.price.historical(symbol=ETF_SYMBOL, provider=PROVIDER, start_date="2024-01-01")),
    run_openbb_route("etf.historical", lambda: obb.etf.historical(symbol=ETF_SYMBOL, provider=PROVIDER, start_date="2024-01-01")),
    run_openbb_route("equity.profile", lambda: obb.equity.profile(symbol=ETF_SYMBOL, provider=PROVIDER)),
    run_openbb_route("etf.info", lambda: obb.etf.info(symbol=ETF_SYMBOL, provider=PROVIDER)),
])

<!-- quant-trader-eda -->
## Quant Trader EDA

ETF checks: return/risk profile, benchmark tracking, holding concentration, and sector exposure.

In [ ]:
def price_eda(frame: pd.DataFrame, annualization: int = 252) -> tuple[pd.DataFrame, pd.DataFrame]:
    if frame is None or frame.empty or "close" not in frame.columns:
        return pd.DataFrame(), pd.DataFrame()
    close = pd.to_numeric(frame["close"], errors="coerce").dropna()
    returns = close.pct_change().dropna()
    if returns.empty:
        return pd.DataFrame(), pd.DataFrame()
    equity_curve = (1 + returns).cumprod()
    drawdown = equity_curve / equity_curve.cummax() - 1
    summary = pd.DataFrame(
        {
            "observations": [int(len(returns))],
            "start": [close.index.min()],
            "end": [close.index.max()],
            "total_return": [close.iloc[-1] / close.iloc[0] - 1],
            "annualized_return": [(1 + returns.mean()) ** annualization - 1],
            "annualized_volatility": [returns.std() * annualization ** 0.5],
            "sharpe_0_rf": [returns.mean() / returns.std() * annualization ** 0.5 if returns.std() else None],
            "max_drawdown": [drawdown.min()],
            "best_day": [returns.max()],
            "worst_day": [returns.min()],
        }
    )
    diagnostics = pd.DataFrame(
        {
            "close": close,
            "return": returns.reindex(close.index),
            "rolling_21d_vol": returns.rolling(21).std() * annualization ** 0.5,
            "rolling_63d_vol": returns.rolling(63).std() * annualization ** 0.5,
            "drawdown": drawdown.reindex(close.index),
        }
    )
    return summary, diagnostics

In [ ]:
etf_prices = warehouse.etf.read_prices(ETF_SYMBOL, provider=PROVIDER)
etf_summary, etf_diagnostics = price_eda(etf_prices)
etf_summary

In [ ]:
preview(etf_diagnostics[["close", "rolling_21d_vol", "rolling_63d_vol", "drawdown"]], rows=10)

In [ ]:
holdings = warehouse.etf.read_fundamentals(ETF_SYMBOL, section="etf_holdings", provider=PROVIDER)
if holdings.empty:
    concentration = pd.DataFrame()
else:
    weight_col = next((column for column in ["weight", "weight_percentage", "percent", "market_value_percentage"] if column in holdings.columns), None)
    name_col = next((column for column in ["symbol", "holding_symbol", "name", "holding_name"] if column in holdings.columns), holdings.columns[0])
    if weight_col is None:
        concentration = holdings.tail(20)
    else:
        latest = holdings.loc[holdings.index == holdings.index.max()].copy() if isinstance(holdings.index, pd.DatetimeIndex) else holdings.copy()
        latest[weight_col] = pd.to_numeric(latest[weight_col], errors="coerce")
        top = latest.sort_values(weight_col, ascending=False).head(10)
        concentration = pd.DataFrame({
            "top_10_weight": [top[weight_col].sum()],
            "holding_count": [len(latest)],
            "largest_weight": [top[weight_col].iloc[0] if not top.empty else None],
        })
        display(top[[name_col, weight_col]].head(10))
concentration

In [ ]:
sectors = warehouse.etf.read_fundamentals(ETF_SYMBOL, section="etf_sectors", provider=PROVIDER)
if sectors.empty:
    pd.DataFrame()
else:
    weight_col = next((column for column in ["weight", "weight_percentage", "percent", "exposure"] if column in sectors.columns), None)
    sector_col = next((column for column in ["sector", "name", "industry"] if column in sectors.columns), sectors.columns[0])
    latest = sectors.loc[sectors.index == sectors.index.max()].copy() if isinstance(sectors.index, pd.DatetimeIndex) else sectors.copy()
    if weight_col is None:
        latest.head(10)
    else:
        latest[weight_col] = pd.to_numeric(latest[weight_col], errors="coerce")
        latest[[sector_col, weight_col]].sort_values(weight_col, ascending=False).head(10)

<!-- quant-trader-signal-eda -->
## Signal Research for P&L

ETF research usually starts with trend, volatility targeting, tracking error, and concentration risk before any allocation decision.

In [ ]:
def signal_backtest(close: pd.Series, signal: pd.Series, annualization: int = 252) -> pd.DataFrame:
    close = pd.to_numeric(close, errors="coerce").dropna()
    returns = close.pct_change().dropna()
    aligned_signal = signal.reindex(returns.index).shift(1).fillna(0).clip(-1, 1)
    strategy_returns = aligned_signal * returns
    if strategy_returns.empty:
        return pd.DataFrame()
    curve = (1 + strategy_returns).cumprod()
    buy_hold = (1 + returns).cumprod()
    drawdown = curve / curve.cummax() - 1
    turnover = aligned_signal.diff().abs().fillna(aligned_signal.abs())
    return pd.DataFrame({
        "strategy_total_return": [curve.iloc[-1] - 1],
        "buy_hold_total_return": [buy_hold.iloc[-1] - 1],
        "strategy_annualized_return": [(1 + strategy_returns.mean()) ** annualization - 1],
        "strategy_annualized_volatility": [strategy_returns.std() * annualization ** 0.5],
        "strategy_sharpe_0_rf": [strategy_returns.mean() / strategy_returns.std() * annualization ** 0.5 if strategy_returns.std() else None],
        "strategy_max_drawdown": [drawdown.min()],
        "hit_rate": [(strategy_returns > 0).mean()],
        "average_exposure": [aligned_signal.abs().mean()],
        "average_daily_turnover": [turnover.mean()],
        "latest_signal": [aligned_signal.iloc[-1]],
    })

In [ ]:
etf_prices = warehouse.etf.read_prices(ETF_SYMBOL, provider=PROVIDER)
if etf_prices.empty or "close" not in etf_prices.columns:
    pd.DataFrame()
else:
    close = etf_prices["close"]
    trend_signal = (close > close.rolling(200).mean()).astype(float)
    signal_backtest(close, trend_signal)

In [ ]:
if etf_prices.empty or "close" not in etf_prices.columns:
    pd.DataFrame()
else:
    close = etf_prices["close"]
    returns = close.pct_change()
    vol_21 = returns.rolling(21).std() * 252 ** 0.5
    allocation = pd.DataFrame({
        "close": close,
        "momentum_63d": close.pct_change(63),
        "momentum_252d": close.pct_change(252),
        "realized_vol_21d": vol_21,
        "vol_target_weight_10pct_cap_1x": (0.10 / vol_21).clip(0, 1),
        "drawdown": close / close.cummax() - 1,
    })
    preview(allocation, rows=15)

In [ ]:
holdings = warehouse.etf.read_fundamentals(ETF_SYMBOL, section="etf_holdings", provider=PROVIDER)
if holdings.empty:
    pd.DataFrame()
else:
    weight_col = next((column for column in ["weight", "weight_percentage", "percent", "market_value_percentage"] if column in holdings.columns), None)
    name_col = next((column for column in ["symbol", "holding_symbol", "name", "holding_name"] if column in holdings.columns), holdings.columns[0])
    latest = holdings.loc[holdings.index == holdings.index.max()].copy() if isinstance(holdings.index, pd.DatetimeIndex) else holdings.copy()
    if weight_col is None:
        latest.head(20)
    else:
        latest[weight_col] = pd.to_numeric(latest[weight_col], errors="coerce")
        latest = latest.sort_values(weight_col, ascending=False)
        pd.DataFrame({
            "top_1_weight": [latest[weight_col].head(1).sum()],
            "top_5_weight": [latest[weight_col].head(5).sum()],
            "top_10_weight": [latest[weight_col].head(10).sum()],
            "effective_positions_herfindahl": [1 / (latest[weight_col].dropna().pow(2).sum()) if latest[weight_col].dropna().pow(2).sum() else None],
            "largest_holding": [latest[name_col].iloc[0] if not latest.empty else None],
        })

<!-- output-analysis -->
## Analysis Notes From This Run

For `SPY`, the stored FMP history is above its 200-day average by about 11.5%, with latest 21-day realized volatility around 9.8%. That is a favorable trend/risk regime for a benchmark exposure model. The simple 200-day trend filter has lower max drawdown than buy-and-hold in this run, which is useful as a risk overlay candidate.

The holdings data shows about 505 holdings, with roughly 26.0% in the top 5 and 37.2% in the top 10. That concentration matters: even broad ETF exposure can be heavily driven by the largest constituents, so single-name and sector concentration should be checked before using `SPY` as a neutral market basket.